# Day 3：手写分布式训练核心组件（本周核心实践 I！）

> **目标**：从零实现分布式训练的核心组件——Ring AllReduce 通信模拟、数据并行训练循环、ZeRO Stage 1/2 优化器分片、Tensor Parallelism 列切分与行切分；通过代码深入理解 Day 1-2 的理论。

**实现路线图**：

```
Part 1: 环境与配置
  ↓
Part 2: 手写 Ring AllReduce（通信模拟）
  ↓
Part 3: 手写数据并行训练循环
  ↓
Part 4: 手写 ZeRO Stage 1（优化器分片）
  ↓
Part 5: 手写 ZeRO Stage 2（梯度 + 优化器分片）
  ↓
Part 6: 手写列并行 Linear（Column Parallel）
  ↓
Part 7: 手写行并行 Linear（Row Parallel）
  ↓
Part 8: TP 应用于 MLP 的完整示例
  ↓
Part 9: 显存分析与验证
  ↓
Part 10: 总结
```

## Part 1：环境与配置

我们使用 Python list 模拟多 GPU 环境，每个 "GPU" 用一个 tensor 表示其持有的数据。
这样可以在单卡（甚至 CPU）环境下理解分布式算法的核心逻辑。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import copy
from dataclasses import dataclass
from typing import List, Optional, Tuple

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

NUM_GPUS = 4  # 模拟 4 张 GPU
print(f"Simulating {NUM_GPUS} GPUs")

## Part 2：手写 Ring AllReduce

Ring AllReduce 分两个阶段：
1. **ReduceScatter**：经过 $N-1$ 步环形传递，每个 GPU 拥有 $1/N$ 的完整归约结果
2. **AllGather**：再经过 $N-1$ 步，每个 GPU 拥有完整的归约结果

每 GPU 总通信量：$2 \times \frac{N-1}{N} \times M$

In [ ]:
def ring_allreduce(gpu_data: List[torch.Tensor]) -> List[torch.Tensor]:
    """
    模拟 Ring AllReduce：对所有 GPU 上的 tensor 求和。
    
    Args:
        gpu_data: 每个 GPU 持有的 tensor，形状相同
    Returns:
        每个 GPU 都持有所有 tensor 之和
    """
    n = len(gpu_data)
    data_size = gpu_data[0].numel()
    chunk_size = data_size // n
    
    # 将每个 GPU 的数据分成 n 个 chunk
    chunks = [d.view(n, chunk_size).clone() for d in gpu_data]
    
    # Phase 1: ReduceScatter
    # 经过 n-1 步，每个 GPU 拥有一个 chunk 的完整归约结果
    for step in range(n - 1):
        for gpu_id in range(n):
            send_chunk_idx = (gpu_id - step) % n
            recv_chunk_idx = (gpu_id - step - 1) % n
            next_gpu = (gpu_id + 1) % n
            # GPU gpu_id 发送 send_chunk_idx 给 next_gpu
            # next_gpu 接收并累加到 recv_chunk_idx
            chunks[next_gpu][recv_chunk_idx] += chunks[gpu_id][send_chunk_idx]
    
    # 此时 GPU i 的 chunk[(i+1)%n] 包含了该 chunk 的全局求和
    
    # Phase 2: AllGather
    # 将每个 GPU 的归约结果广播给所有 GPU
    for step in range(n - 1):
        for gpu_id in range(n):
            send_chunk_idx = (gpu_id - step + 1) % n
            recv_chunk_idx = (gpu_id - step) % n
            next_gpu = (gpu_id + 1) % n
            chunks[next_gpu][recv_chunk_idx] = chunks[gpu_id][send_chunk_idx].clone()
    
    return [c.view(-1) for c in chunks]


# 验证 Ring AllReduce
print("=" * 50)
print("Ring AllReduce 验证")
print("=" * 50)

gpu_data = [torch.arange(8, dtype=torch.float32) + i * 10 for i in range(NUM_GPUS)]
print("\n各 GPU 原始数据:")
for i, d in enumerate(gpu_data):
    print(f"  GPU {i}: {d.tolist()}")

result = ring_allreduce(gpu_data)
expected = sum(gpu_data)

print(f"\n期望结果 (全局求和): {expected.tolist()}")
print("\n AllReduce 后各 GPU 数据:")
for i, d in enumerate(result):
    print(f"  GPU {i}: {d.tolist()}")
    assert torch.allclose(d, expected), f"GPU {i} result mismatch!"

print("\n✅ Ring AllReduce 验证通过！每个 GPU 都拥有完整的归约结果。")

## Part 3：手写数据并行训练循环

数据并行的核心步骤：
1. 每张 GPU 持有**完整模型副本**
2. 将 mini-batch 均分到各 GPU
3. 各 GPU 独立前向 + 反向
4. AllReduce 同步梯度（求平均）
5. 各 GPU 用相同梯度更新参数

数学等价性：$g_{\text{DP}} = \frac{1}{N_d}\sum_{k=1}^{N_d} g_k = g$

In [ ]:
class SimpleModel(nn.Module):
    """用于演示的简单模型"""
    def __init__(self, input_dim=16, hidden_dim=32, output_dim=4):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))


def simulate_data_parallel_step(model: nn.Module, data: torch.Tensor, 
                                 targets: torch.Tensor, num_gpus: int,
                                 lr: float = 0.01):
    """
    模拟一步数据并行训练。
    
    Args:
        model: 原始模型
        data: 完整的 mini-batch 输入
        targets: 完整的 mini-batch 标签
        num_gpus: 模拟 GPU 数量
        lr: 学习率
    """
    batch_size = data.shape[0]
    per_gpu_batch = batch_size // num_gpus
    
    # Step 1: 每张 GPU 复制完整模型
    gpu_models = [copy.deepcopy(model) for _ in range(num_gpus)]
    
    # Step 2: 均分数据
    gpu_data = [data[i*per_gpu_batch:(i+1)*per_gpu_batch] for i in range(num_gpus)]
    gpu_targets = [targets[i*per_gpu_batch:(i+1)*per_gpu_batch] for i in range(num_gpus)]
    
    # Step 3: 各 GPU 独立前向 + 反向
    gpu_losses = []
    for i in range(num_gpus):
        output = gpu_models[i](gpu_data[i])
        loss = F.mse_loss(output, gpu_targets[i])
        loss.backward()
        gpu_losses.append(loss.item())
    
    # Step 4: AllReduce 梯度（求平均）
    for param_name in dict(gpu_models[0].named_parameters()):
        grads = [dict(m.named_parameters())[param_name].grad for m in gpu_models]
        avg_grad = sum(grads) / num_gpus
        for m in gpu_models:
            dict(m.named_parameters())[param_name].grad = avg_grad.clone()
    
    # Step 5: 各 GPU 独立更新参数
    for m in gpu_models:
        with torch.no_grad():
            for p in m.parameters():
                p -= lr * p.grad
    
    # 验证：所有 GPU 的参数应该相同
    for i in range(1, num_gpus):
        for (n1, p1), (n2, p2) in zip(gpu_models[0].named_parameters(), 
                                       gpu_models[i].named_parameters()):
            assert torch.allclose(p1, p2, atol=1e-6), f"Parameter mismatch at {n1}!"
    
    return gpu_models[0], sum(gpu_losses) / num_gpus


# 测试数据并行
print("=" * 50)
print("数据并行训练验证")
print("=" * 50)

model = SimpleModel()
data = torch.randn(32, 16)   # batch_size=32
targets = torch.randn(32, 4)

for step in range(5):
    model, loss = simulate_data_parallel_step(model, data, targets, NUM_GPUS)
    print(f"  Step {step+1}: loss = {loss:.4f}")

print("\n✅ 数据并行验证通过！所有 GPU 参数保持一致。")

## Part 4：手写 ZeRO Stage 1（优化器状态分片）

ZeRO-1 核心思想：
- 优化器状态（Adam 的 $m, v$ + FP32 参数副本）按参数分组切分到各 GPU
- 每卡只维护 $1/N_d$ 的优化器状态
- 梯度仍然 AllReduce
- 参数更新后 AllGather 同步

每卡显存：$4\Phi + \frac{12\Phi}{N_d}$

In [ ]:
class ZeROStage1Optimizer:
    """
    模拟 ZeRO Stage 1：优化器状态分片的 Adam 优化器。
    每个 "GPU" 只维护一部分参数的优化器状态。
    """
    def __init__(self, params: List[nn.Parameter], num_gpus: int, 
                 lr: float = 1e-3, betas: Tuple[float, float] = (0.9, 0.999),
                 eps: float = 1e-8):
        self.params = list(params)
        self.num_gpus = num_gpus
        self.lr = lr
        self.beta1, self.beta2 = betas
        self.eps = eps
        self.step_count = 0
        
        # 将参数分配给各 GPU
        self.param_assignments = {}  # param_idx -> gpu_id
        for i, p in enumerate(self.params):
            self.param_assignments[i] = i % num_gpus
        
        # 每个 GPU 只为自己负责的参数维护优化器状态
        self.m_states = {}   # {gpu_id: {param_idx: tensor}}
        self.v_states = {}   # {gpu_id: {param_idx: tensor}}
        self.fp32_params = {}  # FP32 参数副本
        
        for gpu_id in range(num_gpus):
            self.m_states[gpu_id] = {}
            self.v_states[gpu_id] = {}
            self.fp32_params[gpu_id] = {}
            for i, p in enumerate(self.params):
                if self.param_assignments[i] == gpu_id:
                    self.m_states[gpu_id][i] = torch.zeros_like(p, dtype=torch.float32)
                    self.v_states[gpu_id][i] = torch.zeros_like(p, dtype=torch.float32)
                    self.fp32_params[gpu_id][i] = p.data.float().clone()
    
    def step(self):
        self.step_count += 1
        
        # Step 1: AllReduce 梯度（模拟：直接用梯度，所有 GPU 看到相同梯度）
        # (在真实 DP 中，梯度已经是 AllReduce 后的结果)
        
        # Step 2: 每个 GPU 只更新自己负责的参数
        for gpu_id in range(self.num_gpus):
            for i in self.m_states[gpu_id]:
                p = self.params[i]
                if p.grad is None:
                    continue
                
                grad = p.grad.float()
                m = self.m_states[gpu_id][i]
                v = self.v_states[gpu_id][i]
                fp32_p = self.fp32_params[gpu_id][i]
                
                # Adam 更新
                m.mul_(self.beta1).add_(grad, alpha=1 - self.beta1)
                v.mul_(self.beta2).addcmul_(grad, grad, value=1 - self.beta2)
                
                # Bias correction
                m_hat = m / (1 - self.beta1 ** self.step_count)
                v_hat = v / (1 - self.beta2 ** self.step_count)
                
                fp32_p.add_(m_hat / (v_hat.sqrt() + self.eps), alpha=-self.lr)
                
                self.fp32_params[gpu_id][i] = fp32_p
        
        # Step 3: AllGather — 将更新后的参数同步回所有 GPU
        for gpu_id in range(self.num_gpus):
            for i in self.fp32_params[gpu_id]:
                self.params[i].data = self.fp32_params[gpu_id][i].to(self.params[i].dtype)
    
    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()
    
    def memory_report(self):
        """报告每个 GPU 的优化器状态显存"""
        print("\nZeRO-1 优化器状态分布:")
        for gpu_id in range(self.num_gpus):
            n_params = sum(self.m_states[gpu_id][i].numel() 
                          for i in self.m_states[gpu_id])
            # 每个参数: m(4B) + v(4B) + fp32_copy(4B) = 12 bytes
            mem_bytes = n_params * 12
            print(f"  GPU {gpu_id}: {len(self.m_states[gpu_id])} param groups, "
                  f"{n_params} elements, {mem_bytes / 1024:.1f} KB optimizer states")


# 验证 ZeRO Stage 1
print("=" * 50)
print("ZeRO Stage 1 验证")
print("=" * 50)

model_z1 = SimpleModel()
zero1_opt = ZeROStage1Optimizer(model_z1.parameters(), num_gpus=NUM_GPUS, lr=0.01)
zero1_opt.memory_report()

# 对比：标准 Adam 需要为所有参数维护 m, v
total_params = sum(p.numel() for p in model_z1.parameters())
print(f"\n模型总参数量: {total_params}")
print(f"标准 Adam 优化器状态: {total_params * 12} bytes ({total_params * 12 / 1024:.1f} KB)")
print(f"ZeRO-1 每卡优化器状态: ~{total_params * 12 // NUM_GPUS} bytes "
      f"({total_params * 12 / NUM_GPUS / 1024:.1f} KB)")
print(f"节省: {(1 - 1/NUM_GPUS) * 100:.1f}%")

# 训练几步验证正确性
for step in range(5):
    zero1_opt.zero_grad()
    out = model_z1(torch.randn(8, 16))
    loss = F.mse_loss(out, torch.randn(8, 4))
    loss.backward()
    zero1_opt.step()
    print(f"  Step {step+1}: loss = {loss.item():.4f}")

print("\n✅ ZeRO Stage 1 验证通过！")

## Part 5：手写 ZeRO Stage 2（梯度 + 优化器分片）

ZeRO-2 在 ZeRO-1 基础上，进一步将**梯度也分片**：
- 反向传播时使用 ReduceScatter（而非 AllReduce），每卡只保留 $1/N_d$ 的归约后梯度
- 每卡显存：$2\Phi + \frac{14\Phi}{N_d}$
- 通信量与标准 DP 相同：$2\Phi$

In [ ]:
class ZeROStage2Optimizer:
    """
    模拟 ZeRO Stage 2：梯度 + 优化器状态分片。
    与 ZeRO-1 的区别：反向传播时使用 ReduceScatter 而非 AllReduce，
    每卡只保留自己负责部分的梯度。
    """
    def __init__(self, params: List[nn.Parameter], num_gpus: int,
                 lr: float = 1e-3, betas: Tuple[float, float] = (0.9, 0.999),
                 eps: float = 1e-8):
        self.params = list(params)
        self.num_gpus = num_gpus
        self.lr = lr
        self.beta1, self.beta2 = betas
        self.eps = eps
        self.step_count = 0
        
        # 将参数分配给各 GPU（按参数粒度）
        self.param_assignments = {}
        for i, p in enumerate(self.params):
            self.param_assignments[i] = i % num_gpus
        
        # 优化器状态（同 ZeRO-1）
        self.m_states = {gpu: {} for gpu in range(num_gpus)}
        self.v_states = {gpu: {} for gpu in range(num_gpus)}
        self.fp32_params = {gpu: {} for gpu in range(num_gpus)}
        
        for i, p in enumerate(self.params):
            gpu_id = self.param_assignments[i]
            self.m_states[gpu_id][i] = torch.zeros_like(p, dtype=torch.float32)
            self.v_states[gpu_id][i] = torch.zeros_like(p, dtype=torch.float32)
            self.fp32_params[gpu_id][i] = p.data.float().clone()
    
    def step(self):
        self.step_count += 1
        
        # Step 1: ReduceScatter 梯度
        # 模拟：每卡只保留自己负责的参数的梯度（已归约）
        reduced_grads = {gpu: {} for gpu in range(self.num_gpus)}
        for i, p in enumerate(self.params):
            if p.grad is None:
                continue
            gpu_id = self.param_assignments[i]
            # 在 ReduceScatter 后，只有负责该参数的 GPU 保留梯度
            reduced_grads[gpu_id][i] = p.grad.float().clone()
            # 其他 GPU 可以释放该参数的梯度 → 节省显存
        
        # Step 2: 每个 GPU 只更新自己负责的参数
        for gpu_id in range(self.num_gpus):
            for i in self.m_states[gpu_id]:
                if i not in reduced_grads[gpu_id]:
                    continue
                
                grad = reduced_grads[gpu_id][i]
                m = self.m_states[gpu_id][i]
                v = self.v_states[gpu_id][i]
                fp32_p = self.fp32_params[gpu_id][i]
                
                m.mul_(self.beta1).add_(grad, alpha=1 - self.beta1)
                v.mul_(self.beta2).addcmul_(grad, grad, value=1 - self.beta2)
                
                m_hat = m / (1 - self.beta1 ** self.step_count)
                v_hat = v / (1 - self.beta2 ** self.step_count)
                
                fp32_p.add_(m_hat / (v_hat.sqrt() + self.eps), alpha=-self.lr)
                self.fp32_params[gpu_id][i] = fp32_p
        
        # Step 3: AllGather — 同步更新后的参数
        for gpu_id in range(self.num_gpus):
            for i in self.fp32_params[gpu_id]:
                self.params[i].data = self.fp32_params[gpu_id][i].to(self.params[i].dtype)
    
    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()
    
    def memory_report(self):
        print("\nZeRO-2 显存分布 (优化器 + 梯度):")
        for gpu_id in range(self.num_gpus):
            n_params = sum(self.m_states[gpu_id][i].numel()
                          for i in self.m_states[gpu_id])
            opt_mem = n_params * 12   # m + v + fp32_copy
            grad_mem = n_params * 4   # FP32 梯度（只保留负责的部分）
            print(f"  GPU {gpu_id}: optimizer={opt_mem/1024:.1f} KB, "
                  f"grad={grad_mem/1024:.1f} KB")


# 验证 ZeRO Stage 2
print("=" * 50)
print("ZeRO Stage 2 验证")
print("=" * 50)

model_z2 = SimpleModel()
zero2_opt = ZeROStage2Optimizer(model_z2.parameters(), num_gpus=NUM_GPUS, lr=0.01)
zero2_opt.memory_report()

for step in range(5):
    zero2_opt.zero_grad()
    out = model_z2(torch.randn(8, 16))
    loss = F.mse_loss(out, torch.randn(8, 4))
    loss.backward()
    zero2_opt.step()
    print(f"  Step {step+1}: loss = {loss.item():.4f}")

print("\n✅ ZeRO Stage 2 验证通过！")

## Part 6：手写列并行 Linear（Column Parallel）

张量并行（TP）将层内的权重矩阵切分到多张 GPU 上。

**列切分（Column Parallel）**：

对于线性层 $Y = XA + b$，将权重 $A \in \mathbb{R}^{d_{\text{in}} \times d_{\text{out}}}$ 按列切分：

$$A = [A_1 | A_2 | \cdots | A_N], \quad A_i \in \mathbb{R}^{d_{\text{in}} \times d_{\text{out}}/N}$$

$$Y_i = X A_i \quad \text{(每个 GPU 独立计算一部分输出)}$$

$$Y = [Y_1 | Y_2 | \cdots | Y_N]$$

特点：输入 $X$ 不需要切分（每卡都有完整输入），输出按列拼接。

In [ ]:
class ColumnParallelLinear(nn.Module):
    """
    列并行线性层：将权重按输出维度（列）均匀切分到 N 个 GPU。
    每个 GPU 持有 W[:, start:end] 和 b[start:end]。
    """
    def __init__(self, in_features: int, out_features: int, 
                 num_gpus: int, gpu_id: int, bias: bool = True):
        super().__init__()
        assert out_features % num_gpus == 0
        self.in_features = in_features
        self.out_features_per_gpu = out_features // num_gpus
        self.num_gpus = num_gpus
        self.gpu_id = gpu_id
        
        # 每个 GPU 只持有 1/N 的权重（按列切分）
        self.weight = nn.Parameter(
            torch.randn(self.out_features_per_gpu, in_features) * 0.02
        )
        self.bias = nn.Parameter(
            torch.zeros(self.out_features_per_gpu)
        ) if bias else None
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, in_features) — 每个 GPU 都有完整输入
        # output: (batch, out_features_per_gpu) — 每个 GPU 只有部分输出
        return F.linear(x, self.weight, self.bias)


# 验证列切分
print("=" * 50)
print("Column Parallel Linear 验证")
print("=" * 50)

in_dim, out_dim = 16, 32

# 创建原始的完整 Linear
full_linear = nn.Linear(in_dim, out_dim)

# 创建列并行版本
col_linears = []
for gpu_id in range(NUM_GPUS):
    col_linear = ColumnParallelLinear(in_dim, out_dim, NUM_GPUS, gpu_id)
    # 从完整权重中复制对应列
    start = gpu_id * (out_dim // NUM_GPUS)
    end = start + (out_dim // NUM_GPUS)
    col_linear.weight.data = full_linear.weight.data[start:end].clone()
    col_linear.bias.data = full_linear.bias.data[start:end].clone()
    col_linears.append(col_linear)

# 测试
x = torch.randn(4, in_dim)
full_output = full_linear(x)
partial_outputs = [cl(x) for cl in col_linears]
concat_output = torch.cat(partial_outputs, dim=-1)  # AllGather

print(f"输入形状: {x.shape}")
print(f"完整 Linear 输出: {full_output.shape}")
for i, po in enumerate(partial_outputs):
    print(f"GPU {i} 部分输出: {po.shape}")
print(f"拼接后输出: {concat_output.shape}")
print(f"最大差异: {(full_output - concat_output).abs().max().item():.2e}")
assert torch.allclose(full_output, concat_output, atol=1e-5)
print("\n✅ Column Parallel Linear 验证通过！")

## Part 7：手写行并行 Linear（Row Parallel）

**行切分（Row Parallel）**：

将权重 $A$ 按行切分：

$$A = \begin{bmatrix} A_1 \\ A_2 \\ \vdots \\ A_N \end{bmatrix}, \quad A_i \in \mathbb{R}^{d_{\text{in}}/N \times d_{\text{out}}}$$

对应地，输入 $X$ 也需要按列切分：

$$Y = XA = [X_1 | X_2 | \cdots | X_N] \begin{bmatrix} A_1 \\ \vdots \\ A_N \end{bmatrix} = \sum_{i=1}^{N} X_i A_i$$

每个 GPU 计算 $X_i A_i$，然后 **AllReduce（求和）** 得到完整输出。

In [ ]:
class RowParallelLinear(nn.Module):
    """
    行并行线性层：将权重按输入维度（行）均匀切分到 N 个 GPU。
    每个 GPU 持有 W[start:end, :]。
    输入也需要按列切分，输出通过 AllReduce 求和。
    """
    def __init__(self, in_features: int, out_features: int,
                 num_gpus: int, gpu_id: int, bias: bool = True):
        super().__init__()
        assert in_features % num_gpus == 0
        self.in_features_per_gpu = in_features // num_gpus
        self.out_features = out_features
        self.num_gpus = num_gpus
        self.gpu_id = gpu_id
        
        # 每个 GPU 只持有 1/N 的权重（按行切分）
        self.weight = nn.Parameter(
            torch.randn(out_features, self.in_features_per_gpu) * 0.02
        )
        # bias 只在一个 GPU 上，避免重复累加
        self.bias = nn.Parameter(
            torch.zeros(out_features)
        ) if (bias and gpu_id == 0) else None
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, in_features_per_gpu) — 每个 GPU 只接收部分输入
        # output: (batch, out_features) — 部分结果，需要 AllReduce 求和
        out = F.linear(x, self.weight)
        if self.bias is not None:
            out = out + self.bias
        return out


# 验证行切分
print("=" * 50)
print("Row Parallel Linear 验证")
print("=" * 50)

in_dim, out_dim = 32, 16
full_linear = nn.Linear(in_dim, out_dim)

row_linears = []
for gpu_id in range(NUM_GPUS):
    row_linear = RowParallelLinear(in_dim, out_dim, NUM_GPUS, gpu_id, bias=(gpu_id==0))
    start = gpu_id * (in_dim // NUM_GPUS)
    end = start + (in_dim // NUM_GPUS)
    row_linear.weight.data = full_linear.weight.data[:, start:end].clone()
    if gpu_id == 0:
        row_linear.bias.data = full_linear.bias.data.clone()
    row_linears.append(row_linear)

x = torch.randn(4, in_dim)
full_output = full_linear(x)

# 每个 GPU 接收切分后的输入
partial_results = []
for gpu_id in range(NUM_GPUS):
    start = gpu_id * (in_dim // NUM_GPUS)
    end = start + (in_dim // NUM_GPUS)
    x_part = x[:, start:end]  # 切分输入
    partial_results.append(row_linears[gpu_id](x_part))

# AllReduce（求和）得到完整输出
allreduced_output = sum(partial_results)

print(f"输入形状: {x.shape}")
print(f"完整 Linear 输出: {full_output.shape}")
for i, pr in enumerate(partial_results):
    print(f"GPU {i} 部分结果: {pr.shape}")
print(f"AllReduce 后输出: {allreduced_output.shape}")
print(f"最大差异: {(full_output - allreduced_output).abs().max().item():.2e}")
assert torch.allclose(full_output, allreduced_output, atol=1e-5)
print("\n✅ Row Parallel Linear 验证通过！")

## Part 8：TP 应用于 MLP 的完整示例

Megatron-LM 中 MLP 的 TP 策略：将两个 Linear 层分别用列切分和行切分，
中间不需要通信，只在最后做一次 AllReduce：

```
输入 X (每卡完整)
    │
    ├── GPU 0: X @ A₁ → GeLU → (·) @ B₁  ──┐
    ├── GPU 1: X @ A₂ → GeLU → (·) @ B₂  ──┤ AllReduce (求和)
    ├── GPU 2: X @ A₃ → GeLU → (·) @ B₃  ──┤
    └── GPU 3: X @ A₄ → GeLU → (·) @ B₄  ──┘
                                              │
                                          输出 Y (每卡完整)
```

关键：GeLU 是逐元素操作，可以在列切分后独立计算。
第一个 Linear 用列切分，第二个用行切分 → 只需 1 次 AllReduce。

In [ ]:
class TensorParallelMLP(nn.Module):
    """
    张量并行 MLP：第一个 Linear 列切分，第二个 Linear 行切分。
    模拟单个 GPU 视角的计算。
    """
    def __init__(self, d_model: int, d_ff: int, num_gpus: int, gpu_id: int):
        super().__init__()
        assert d_ff % num_gpus == 0
        self.gpu_id = gpu_id
        self.num_gpus = num_gpus
        
        # 第一个 Linear: 列切分 (d_model → d_ff/N)
        self.w1 = ColumnParallelLinear(d_model, d_ff, num_gpus, gpu_id, bias=True)
        # 第二个 Linear: 行切分 (d_ff/N → d_model)
        self.w2 = RowParallelLinear(d_ff, d_model, num_gpus, gpu_id, bias=(gpu_id==0))
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, d_model) — 完整输入
        h = self.w1(x)              # (batch, d_ff/N) — 列切分输出
        h = F.gelu(h)               # GeLU 可以独立计算（逐元素）
        out = self.w2(h)            # (batch, d_model) — 部分结果
        return out                  # 需要 AllReduce 求和


# 验证 TP MLP
print("=" * 50)
print("Tensor Parallel MLP 验证")
print("=" * 50)

d_model, d_ff = 64, 256

# 创建完整 MLP 作为参照
class FullMLP(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff)
        self.w2 = nn.Linear(d_ff, d_model)
    def forward(self, x):
        return self.w2(F.gelu(self.w1(x)))

full_mlp = FullMLP(d_model, d_ff)

# 创建 TP MLP 并复制权重
tp_mlps = []
for gpu_id in range(NUM_GPUS):
    tp_mlp = TensorParallelMLP(d_model, d_ff, NUM_GPUS, gpu_id)
    
    # 复制 w1 (列切分) 权重
    ff_per_gpu = d_ff // NUM_GPUS
    start, end = gpu_id * ff_per_gpu, (gpu_id + 1) * ff_per_gpu
    tp_mlp.w1.weight.data = full_mlp.w1.weight.data[start:end].clone()
    tp_mlp.w1.bias.data = full_mlp.w1.bias.data[start:end].clone()
    
    # 复制 w2 (行切分) 权重
    tp_mlp.w2.weight.data = full_mlp.w2.weight.data[:, start:end].clone()
    if gpu_id == 0:
        tp_mlp.w2.bias.data = full_mlp.w2.bias.data.clone()
    
    tp_mlps.append(tp_mlp)

# 测试
x = torch.randn(4, d_model)
full_output = full_mlp(x)

tp_partial = [tp_mlps[i](x) for i in range(NUM_GPUS)]
tp_output = sum(tp_partial)  # AllReduce

print(f"输入: {x.shape}")
print(f"完整 MLP 输出: {full_output.shape}")
print(f"TP MLP 输出 (AllReduce 后): {tp_output.shape}")
print(f"最大差异: {(full_output - tp_output).abs().max().item():.2e}")
assert torch.allclose(full_output, tp_output, atol=1e-4)

# 参数分析
full_params = sum(p.numel() for p in full_mlp.parameters())
tp_params_per_gpu = sum(p.numel() for p in tp_mlps[0].parameters())
print(f"\n完整 MLP 参数量: {full_params}")
print(f"TP 每 GPU 参数量: {tp_params_per_gpu}")
print(f"参数减少: {(1 - tp_params_per_gpu / full_params) * 100:.1f}%")
print(f"\n通信: 1 次 AllReduce (求和)，每 GPU 发送 {x.shape[0] * d_model * 4} bytes")

print("\n✅ Tensor Parallel MLP 验证通过！")

## Part 9：显存分析与验证

汇总对比不同并行策略下的显存占用，验证 Day 1-2 的理论公式。

In [ ]:
def memory_analysis(model_params_B: float, num_gpus: int, 
                     bytes_per_param: int = 2):
    """
    分析不同并行策略下的每卡显存占用。
    
    Args:
        model_params_B: 模型参数量（十亿）
        num_gpus: GPU 数量
        bytes_per_param: 参数精度（FP16=2, FP32=4）
    """
    phi = model_params_B * 1e9  # 总参数量
    Nd = num_gpus
    
    # 各组成的每参数字节数 (混合精度 AdamW)
    param_bytes = 2    # FP16
    grad_bytes = 2     # FP16
    opt_bytes = 12     # FP32 copy(4) + m(4) + v(4)
    total_bytes = param_bytes + grad_bytes + opt_bytes  # = 16
    
    results = {}
    
    # DP (无优化)
    results['DP'] = total_bytes * phi / (1024**3)
    
    # ZeRO Stage 1: 优化器分片
    results['ZeRO-1'] = ((param_bytes + grad_bytes) * phi + opt_bytes * phi / Nd) / (1024**3)
    
    # ZeRO Stage 2: 梯度 + 优化器分片
    results['ZeRO-2'] = (param_bytes * phi + (grad_bytes + opt_bytes) * phi / Nd) / (1024**3)
    
    # ZeRO Stage 3: 全分片
    results['ZeRO-3'] = total_bytes * phi / Nd / (1024**3)
    
    # TP (参数切分)
    results['TP only'] = total_bytes * phi / Nd / (1024**3)
    
    return results


print("=" * 70)
print("显存分析对比表")
print("=" * 70)

for model_name, model_size in [("LLaMA-7B", 7), ("LLaMA-13B", 13), ("LLaMA-70B", 70)]:
    print(f"\n{model_name} ({model_size}B params), AdamW + FP16 混合精度:")
    for num_gpus in [8, 64]:
        results = memory_analysis(model_size, num_gpus)
        print(f"  {num_gpus} GPUs:")
        for strategy, mem in results.items():
            feasible = "✅" if mem <= 80 else "❌"
            print(f"    {strategy:12s}: {mem:8.1f} GB / GPU  {feasible} (A100 80GB)")

# 通信量分析
print("\n" + "=" * 70)
print("通信量分析 (7B 模型, FP16)")
print("=" * 70)

phi_7b = 7e9
comm_dp = 2 * phi_7b * 2 / (1024**3)  # AllReduce: 2M
comm_z1 = 3 * phi_7b * 2 / (1024**3)  # AllReduce + AllGather: 3M
comm_z2 = 2 * phi_7b * 2 / (1024**3)  # ReduceScatter + AllGather: 2M
comm_z3 = 3 * phi_7b * 2 / (1024**3)  # 2×AllGather + ReduceScatter: 3M

print(f"  DP:     {comm_dp:.1f} GB (AllReduce)")
print(f"  ZeRO-1: {comm_z1:.1f} GB (AllReduce + AllGather)")
print(f"  ZeRO-2: {comm_z2:.1f} GB (ReduceScatter + AllGather) = DP 相同!")
print(f"  ZeRO-3: {comm_z3:.1f} GB (2×AllGather + ReduceScatter) = 1.5× DP")

## Part 10：总结

本 notebook 实现了分布式训练的三大核心组件：

| 组件 | 实现 | 核心思想 |
|------|------|----------|
| Ring AllReduce | `ring_allreduce()` | 环形通信，每 GPU 通信量 $\approx 2M$，与 GPU 数无关 |
| 数据并行 (DP) | `simulate_data_parallel_step()` | 数据切分 + 梯度 AllReduce，数学等价大 batch |
| ZeRO Stage 1 | `ZeROStage1Optimizer` | 优化器状态分片，每卡 $4\Phi + 12\Phi/N_d$ |
| ZeRO Stage 2 | `ZeROStage2Optimizer` | 梯度+优化器分片，每卡 $2\Phi + 14\Phi/N_d$ |
| Column Parallel | `ColumnParallelLinear` | 权重按列切分，输出按列拼接 |
| Row Parallel | `RowParallelLinear` | 权重按行切分，输出 AllReduce 求和 |
| TP MLP | `TensorParallelMLP` | 列切分 → GeLU → 行切分 → 1 次 AllReduce |

**面试重点**：能画出 ZeRO 三阶段显存对比图，能写出 TP 列切分/行切分的数学公式和 MLP 数据流。